# Metabolomics VIP analyses

Date last updated: 1/26/26

Notebook author: Yang Chen, Britta De Pessemier

Data analysis by: Britta De Pessemier, Yang Chen

This notebook plots the following:
- Top VIPs separating between pairwise skin groups and univariate statistical testing

In [47]:
# Import Python packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.ticker as ticker
from matplotlib.patches import Rectangle
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import Normalize
from brokenaxes import brokenaxes
from matplotlib.colors import LinearSegmentedColormap
import biom
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
from pathlib import Path
from sklearn.utils import resample


### Import and organize VIP results

In [48]:
# Base path for metabolomics output
BASE = Path('../Data/metabolomics/Run3_10252024/output')

# Mapping to harmonize GroupContrib labels across contrasts
GROUP_CONTRIB_MAP = {
    'Acne Lesional': 'Acne_L',
    'Acne Non-lesional': 'Acne_NL',
    'Healthy': 'Healthy'
}

# Helper function to load a VIP table and annotate contrast group
def load_vip_table(filename, group_label):
    df = pd.read_csv(BASE / filename)
    df['group'] = group_label
    return df

# Load merged VIP feature tables
VIP_LvsH_merged = load_vip_table(
    'Top_Features_VIP_LvsH_merged_data_method2_11212024.csv',
    'H vs L'
)

VIP_NLvsH_merged = load_vip_table(
    'Top_Features_VIP_NLvsH_merged_data_method2_11212024.csv',
    'H vs NL'
)

VIP_NLvsL_merged = load_vip_table(
    'Top_Features_VIP_NLvsL_merged_data_method2_11212024.csv',
    'NL vs L'
)

# Apply consistent GroupContrib naming across all VIP tables
vip_tables = [
    VIP_LvsH_merged, VIP_NLvsH_merged, VIP_NLvsL_merged
]

for df in vip_tables:
    if 'GroupContrib' in df.columns:
        df['GroupContrib'] = df['GroupContrib'].replace(GROUP_CONTRIB_MAP)

# Optional dictionary for programmatic access to all VIP tables
VIP = {
    'LvsH_merged': VIP_LvsH_merged,
    'NLvsH_merged': VIP_NLvsH_merged,
    'NLvsL_merged': VIP_NLvsL_merged
}

In [49]:
# Concatenate merged VIP feature tables across contrasts
VIP_combined = pd.concat(
    [VIP_LvsH_merged, VIP_NLvsH_merged, VIP_NLvsL_merged],
    ignore_index=True
)

# Sort combined VIP table by component loading magnitude
VIP_sorted = VIP_combined.sort_values(by='comp1', ascending=False)

# Fill missing chemical ontology annotations with 'Unknown'
VIP_sorted[['superclass', 'class', 'subclass']] = (
    VIP_sorted[['superclass', 'class', 'subclass']]
    .fillna('Unknown')
)

# Retain features with VIP score >= 1
VIP_filtered = VIP_sorted[VIP_sorted['comp1'] >= 1]

# Retain only features with an assigned molecular formula
VIP_filtered = VIP_filtered[VIP_filtered['molecular_formula'].notna()]
VIP_filtered

,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,...,InChIKey,InChIKey.Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi,group
18,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,LUIGTZGBXWZJAX-UHFFFAOYSA-N,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL
0,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,LUIGTZGBXWZJAX-UHFFFAOYSA-N,LUIGTZGBXWZJAX,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols,Fatty acyls,Fatty alcohols,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L
45,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,QIVBCDIJIAJPQS-SECBINFHSA-N,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L
20,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,0,...,SNICXCGAKADSCV-JTQLQIEISA-N,SNICXCGAKADSCV,Organoheterocyclic compounds,Pyridines and derivatives,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL
48,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,...,NWGKJDSIEKMTRX-UHFFFAOYSA-N,NWGKJDSIEKMTRX,Lipids and lipid-like molecules,Fatty Acyls,Fatty acid esters,Fatty esters,Wax monoesters,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L
23,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,QIVBCDIJIAJPQS-SECBINFHSA-N,QIVBCDIJIAJPQS,Organoheterocyclic compounds,Indoles and derivatives,Indolyl carboxylic acids and derivatives,Small peptides,Aminoacids,Amino acids and Peptides|Shikimates and Phenyl...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL
49,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,306400,0,...,VVLXCWVSSLFQDS-UHFFFAOYSA-N,VVLXCWVSSLFQDS,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Dipeptides,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005747606,NL vs L
50,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,637600,0,...,LKMNXYDUQXAUCZ-UHFFFAOYSA-N,LKMNXYDUQXAUCZ,Phenylpropanoids and polyketides,Flavonoids,O-methylated flavonoids,Flavonoids,Flavones,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012283450,NL vs L
24,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,282600,0,...,BOWVQLFMWHZBEF-KTKRTIGZSA-N,BOWVQLFMWHZBEF,Organic nitrogen compounds,Organonitrogen compounds,Amines,Fatty amides,N-acyl ethanolamines (endocannabinoids),Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL
52,777,1.758563,Acne_L,0.052337,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,0,...,SNICXCGAKADSCV-JTQLQIEISA-N,SNICXCGAKADSCV,Organoheterocyclic compounds,Pyridines and derivatives,Pyrrolidinylpyridines,Nicotinic acid alkaloids,Pyridine alkaloids,Alkaloids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,NL vs L


In [50]:
# Define a function to calculate the VIP_direction
def calculate_vip_direction(row):
    if row['group'] == 'H vs L':
        return row['comp1'] if row['GroupContrib'] == 'Acne_L' else -row['comp1']
    elif row['group'] == 'H vs NL':
        return row['comp1'] if row['GroupContrib'] == 'Acne_NL' else -row['comp1']
    elif row['group'] == 'NL vs L':
        return row['comp1'] if row['GroupContrib'] == 'Acne_L' else -row['comp1']
    else:
        return row['comp1']  # Default to keeping the value unchanged if no match

# Apply the function to create the new column
VIP_filtered['VIP_direction'] = VIP_filtered.apply(calculate_vip_direction, axis=1)

VIP_filtered = VIP_filtered.dropna(subset=['GroupContrib'])

In [51]:
# read in metadata info on metabolites
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')

In [52]:
# Merge mz info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'mz']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Merge RT info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'RT']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

# Merge ClassyFire info based on matching IDs
VIP_filtered = VIP_filtered.merge(
    info_feature_complete[['Feature', 'ClassyFire#most specific class']],
    left_on='ID',
    right_on='Feature',
    how='left'
)

VIP_filtered.to_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_data_method2_11212024.csv')

In [53]:
VIP_filtered = pd.read_csv('../Data/metabolomics/Run3_10252024/output/VIP_filtered_final_data_method2_11212024_with_shortened_names.csv', index_col=0)
VIP_filtered

,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,...,library_usi,group,VIP_direction,Feature_x,mz,Feature_y,RT,Feature,ClassyFire#most specific class,Shortened_Compound_Name
0,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs NL,2.594219,29059,311.257397,29059,7.421430,29059,Fatty alcohols,C19H36O4 Fatty Alcohol
1,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005722082,H vs L,2.358565,29059,311.257397,29059,7.421430,29059,Fatty alcohols,C19H36O4 Fatty Alcohol
2,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,NL vs L,-2.271726,5930,205.096918,5930,2.076001,5930,Alpha amino acids,Tryptophan
3,777,2.116338,Healthy,0.063125,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,H vs NL,-2.116338,777,163.122416,777,0.634317,777,Pyrrolidinylpyridines,Nicotine
4,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00000577480,NL vs L,1.947722,24392,428.372821,24392,6.156398,24392,Alpha amino acids and derivatives,Polysorbate derivative
5,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883949,H vs NL,1.922795,5930,205.096918,5930,2.076001,5930,Alpha amino acids,Tryptophan
6,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,306400,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005747606,NL vs L,-1.824152,3794,311.123857,3794,1.203634,3794,Gamma-glutamyl amino acids,Glutamyltyrosine
7,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,637600,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012283450,NL vs L,1.823974,17958,373.127569,17958,5.024735,17958,5-O-methylated flavonoids,Sinensetin
8,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,282600,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003137348,H vs NL,1.814812,27758,326.305056,27758,7.132356,27758,N-acyl amines,N-Oleoylethanolamine
9,777,1.758563,Acne_L,0.052337,CCMSLIB00005735151,spectra_filtered.mgf,MASSBANK.mgf,0.981793,3738000,0,...,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005735151,NL vs L,1.758563,777,163.122416,777,0.634317,777,Pyrrolidinylpyridines,Nicotine


### Perform univariate statistical testing on above metabolites w/ top VIPs

In [54]:
# Function to load BIOM table
def load_biom_table(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    df = df.T
    
    return df

In [55]:
# Read in metabolomics table
metaB_table = load_biom_table('../Data/metabolomics/Run3_10252024/metabolomics_method2.biom')

# Convert columns in metaB_table to int (if applicable)
metaB_table.columns = metaB_table.columns.astype(int)

# Convert features_to_keep to int
features_to_keep = VIP_filtered['ID'].unique().astype(int)

# Subset metaB_table using the integer feature list
metaB_table_subset = metaB_table.loc[:, metaB_table.columns.isin(features_to_keep)]


In [56]:
# Read in sample metadata
metadata = pd.read_csv('../Metadata/metadata_final_22102024.tsv', sep='\t')

# Subset metadata based on the indices of metaB_subset
metadata_subset = metadata[metadata['SampleID'].isin(metaB_table_subset.index)]

metadata_subset.head()

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,subject_ID_x_group
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_310_Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL,absent,absent Acne_NL,PP_305_Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L,low,low Acne_L,PP_306_Acne_L
6,LAMI.RD317.D14.C2,C2,70,Lesional,skin,Day 14,44,14,317,15,...,acne,RD317,acne,PP_317,PP_317C2,Lesional_C2,Acne_L,low,low Acne_L,PP_317_Acne_L
7,LAMI.RD305.D23.C1,C1,not applicable,Lesional,skin,Day 23,not applicable,23,305,not applicable,...,acne,RD305,acne,PP_305,PP_305C1,Lesional_C1,Acne_L,low,low Acne_L,PP_305_Acne_L


In [57]:
# Map the 'group' column from metadata_subset to metaB_subset
metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])

# Replace values in the 'group' column
metaB_table_subset['group'] = metaB_table_subset['group'].replace({
    'Healthy': 'H',
    'Acne_L': 'L',
    'Acne_NL': 'NL'
})

# Map the 'subject_ID' column from metadata_subset to metaB_subset
metaB_table_subset['subject_ID'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['subject_ID'])

metaB_table_subset.head()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/1371939033.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset.index.map(metadata_subset.set_index('SampleID')['group'])
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/1371939033.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metaB_table_subset['group'] = metaB_table_subset['group'].replace({
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/1371939033.py:12

,954,1368,941,655,5062,777,2969,1119,5930,3496,...,29059,27758,23297,17958,19710,24392,20967,11925,group,subject_ID
LAMI.RD001.D0.C1,2987383.8,2025607.40,29574.6480,39859.7580,28277.594,0.0,970487.50,1942604.5,567510.06,1473068.10,...,611709.70,16778.8380,0.0000,0.000,0.000,0.00,0.0000,125692.3300,H,PP_1
LAMI.RD308.D2.C2,1121916.4,1669351.50,2905.5796,3065.1714,19280.236,0.0,750870.10,1618142.8,442653.66,286004.00,...,231525.84,14299.3520,0.0000,0.000,0.000,0.00,3822.6377,4734.5693,L,PP_308
LAMI.RD304.D11.C1,1818308.6,1434033.20,47379.2730,36750.4000,15391.807,0.0,595568.94,1605551.0,328151.12,299140.06,...,78491.53,0.0000,5375.8525,11225.194,23546.951,99470.69,29824.6270,12473.0920,L,PP_304
LAMI.RD304.D11.C2,1126503.2,706197.75,3460.0974,0.0000,15015.189,0.0,303845.30,865753.5,152392.72,191713.67,...,261996.80,15780.3090,2691.0256,16378.752,34863.656,157503.92,44084.5860,40673.1100,L,PP_304
LAMI.RD304.D0.C1,827466.3,778566.10,4547.2640,3345.8184,45187.030,0.0,293386.34,1321474.4,137201.39,705570.80,...,427202.06,3228.1143,3431.0034,0.000,0.000,0.00,0.0000,49464.9020,L,PP_304


In [58]:
# Prepare the dictionary to set up univariate testing
feature_group_dict = (
    VIP_filtered.groupby('ID')['group']
    .apply(lambda x: x.unique().tolist())
    .to_dict()
)

# Initialize an empty dictionary to store results
statistical_results = {}

# Iterate through each metabolite in feature_group_dict
for metabolite, comparisons in feature_group_dict.items():
    # Create a dictionary to store results for the current metabolite
    metabolite_results = {}
    
    # Get the values for the metabolite from metaB_table_subset
    metabolite_values = metaB_table_subset[metabolite]
    
    # Iterate through each comparison for the current metabolite
    for comparison in comparisons:
        # Split the comparison into X and Y
        group_x, group_y = comparison.split(" vs ")
        
        # Subset values for the two groups
        values_x = metabolite_values[metaB_table_subset['group'] == group_x]
        values_y = metabolite_values[metaB_table_subset['group'] == group_y]
        
        # Perform the Mann-Whitney U test
        stat, p_value = mannwhitneyu(values_x, values_y, alternative='two-sided')
        
        # Store the result
        metabolite_results[comparison] = {'U-statistic': stat, 'p-value': p_value}
    
    # Add results for the current metabolite to the main results dictionary
    statistical_results[metabolite] = metabolite_results

# Convert results to a DataFrame for easier viewing (optional)
results_df = pd.DataFrame.from_dict({(met, comp): vals 
                                     for met, comps in statistical_results.items() 
                                     for comp, vals in comps.items()}, 
                                    orient='index')


In [59]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='ID')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)

# Create a mapping from Feature to Shortened_Compound_Name
feature_to_name_mapping = VIP_filtered_unique.set_index('ID')['Shortened_Compound_Name']
feature_to_name_mapping = VIP_filtered_unique.set_index('ID')['Compound_Name']

# Map the Shortened_Compound_Name to results_df
results_df['Shortened_Compound_Name'] = feature_index.map(feature_to_name_mapping)
results_df['Compound_Name'] = feature_index.map(feature_to_name_mapping)

results_df


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/2752189583.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)


U-statistic   p-value  \
379   NL vs L       2506.0  0.118156   
      H vs NL        575.0  0.246680   
655   NL vs L       2622.0  0.035492   
      H vs NL        606.0  0.405200   
777   H vs NL        876.5  0.027488   
      NL vs L       2071.0  0.767129   
      H vs L        2942.0  0.016249   
862   NL vs L       2472.0  0.155851   
      H vs NL        590.0  0.319225   
941   H vs L        2218.0  0.526262   
954   H vs NL        551.0  0.156106   
      H vs L        2237.0  0.576631   
1119  NL vs L       2486.0  0.139352   
      H vs NL        639.0  0.641541   
1197  NL vs L       2495.5  0.128945   
      H vs NL        598.0  0.363119   
1368  H vs NL        538.0  0.118988   
      NL vs L       2558.0  0.074771   
      H vs L        2222.0  0.536907   
2969  H vs NL        531.0  0.102086   
      NL vs L       2579.0  0.061424   
      H vs L        2189.0  0.454638   
3496  H vs NL        805.0  0.186551   
3794  NL vs L       2513.0  0.109209   
      H vs L        2275.0  0.680104   
5062  H vs NL        626.0  0.544321   
      H vs L        2367.0  0.963808   
5930  NL vs L       2769.0  0.007546   
      H vs NL        501.0  0.050040   
      H vs L        2193.0  0.464216   
7778  H vs NL        644.0  0.584706   
      H vs L        2513.0  0.431753   
11925 H vs NL        610.0  0.426629   
      H vs L        2264.0  0.644373   
15770 NL vs L       2142.0  0.963042   
17958 NL vs L       2156.0  0.881355   
19710 NL vs L       2136.0  0.997156   
20967 H vs NL        625.0  0.224219   
      NL vs L       2142.0  0.963042   
23297 H vs L        1725.5  0.000763   
      NL vs L       2404.5  0.177976   
24392 NL vs L       1743.0  0.080255   
      H vs NL        669.0  0.872994   
27758 H vs NL        558.0  0.179420   
      H vs L        2232.5  0.564559   
29059 H vs NL        469.0  0.021106   
      H vs L        1864.0  0.042352   

                                         Shortened_Compound_Name  \
379   NL vs L                                            URIDINE   
      H vs NL                                            URIDINE   
655   NL vs L                                      UROCANIC ACID   
      H vs NL                                      UROCANIC ACID   
777   H vs NL  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      NL vs L  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      H vs L   Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
862   NL vs L                                     Cyclo(his-pro)   
      H vs NL                                     Cyclo(his-pro)   
941   H vs L                                       UROCANIC ACID   
954   H vs NL                      L-Pyroglutamic acid - 40.0 eV   
      H vs L                       L-Pyroglutamic acid - 40.0 eV   
1119  NL vs L           Spectral Match to L-Tyrosine from NIST14   
      H vs NL           Spectral Match to L-Tyrosine from NIST14   
1197  NL vs L           Massbank:PR310825 N-Fructosyl isoleucine   
      H vs NL           Massbank:PR310825 N-Fructosyl isoleucine   
1368  H vs NL                               ISOLEUCINE - 60.0 eV   
      NL vs L                               ISOLEUCINE - 60.0 eV   
      H vs L                                ISOLEUCINE - 60.0 eV   
2969  H vs NL                           Phenylalanine - 50.00 eV   
      NL vs L                           Phenylalanine - 50.00 eV   
      H vs L                            Phenylalanine - 50.00 eV   
3496  H vs NL                                       DL-Panthenol   
3794  NL vs L                 Massbank:PR311057 Glutamyltyrosine   
      H vs L                  Massbank:PR311057 Glutamyltyrosine   
5062  H vs NL                              Paracetamol - 40.0 eV   
      H vs L                               Paracetamol - 40.0 eV   
5930  NL vs L                             D-TRYPTOPHAN - 60.0 eV   
      H vs NL                             D-TRYPTOPHAN - 60.0 eV   
      H vs L                              D-TRYPTOPHAN

In [60]:
# Subset to rows where p-value <= 0.05
results_df_sig = results_df[results_df['p-value'] <= 0.05]

# Assign names based on index
results_df_sig.loc[655, 'Final_Name'] = 'Urocanic acid'
results_df_sig.loc[5930, 'Final_Name'] = 'Tryptophan'
results_df_sig.loc[23297, 'Final_Name'] = 'Gln-C14:0'
results_df_sig.loc[29059, 'Final_Name'] = 'C19H36O4 Fatty alcohol'

results_df_sig

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/3626744057.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_df_sig.loc[655, 'Final_Name'] = 'Urocanic acid'


U-statistic   p-value  \
655   NL vs L       2622.0  0.035492   
777   H vs NL        876.5  0.027488   
      H vs L        2942.0  0.016249   
5930  NL vs L       2769.0  0.007546   
23297 H vs L        1725.5  0.000763   
29059 H vs NL        469.0  0.021106   
      H vs L        1864.0  0.042352   

                                         Shortened_Compound_Name  \
655   NL vs L                                      UROCANIC ACID   
777   H vs NL  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      H vs L   Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
5930  NL vs L                             D-TRYPTOPHAN - 60.0 eV   
23297 H vs L                                           Gln-C14:0   
29059 H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   

                                                   Compound_Name  \
655   NL vs L                                      UROCANIC ACID   
777   H vs NL  Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
      H vs L   Massbank:EQ300804 Nicotine|3-[(2S)-1-methylpyr...   
5930  NL vs L                             D-TRYPTOPHAN - 60.0 eV   
23297 H vs L                                           Gln-C14:0   
29059 H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   

                           Final_Name  
655   NL vs L           Urocanic acid  
777   H vs NL                     NaN  
      H vs L                      NaN  
5930  NL vs L              Tryptophan  
23297 H vs L                Gln-C14:0  
29059 H vs NL  C19H36O4 Fatty alcohol  
      H vs L   C19H36O4 Fatty alcohol

In [61]:
# Get list
significant_features = results_df_sig['Final_Name'].dropna().tolist()
significant_features

['Urocanic acid',
 'Tryptophan',
 'Gln-C14:0',
 'C19H36O4 Fatty alcohol',
 'C19H36O4 Fatty alcohol']

In [62]:
# Ensure the Feature column in VIP_filtered has unique mappings
VIP_filtered_unique = VIP_filtered.drop_duplicates(subset='ID')

# Check if results_df has a MultiIndex
if isinstance(results_df.index, pd.MultiIndex):
    # Use the first level of the MultiIndex for mapping
    feature_index = results_df.index.get_level_values(0).astype(int)
else:
    # Use the index directly if not a MultiIndex
    feature_index = results_df.index.astype(int)

# Ensure Feature column in VIP_filtered is of the same type
VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)

# Create a mapping from Feature to VIP values
feature_to_vip_mapping = VIP_filtered_unique.set_index('ID')['comp1']

# Map the VIP values to results_df
results_df['comp1'] = feature_index.map(feature_to_vip_mapping)

# Sort results_df by the 'VIP' column in descending order
results_df = results_df.sort_values(by='comp1', ascending=False)
results_df = results_df.drop((777,), errors='ignore')

filtered_features = results_df
filtered_features.head()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/19809551.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_unique['ID'] = VIP_filtered_unique['ID'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/19809551.py:23: PerformanceWarning: indexing past lexsort depth may impact performance.
  results_df = results_df.drop((777,), errors='ignore')


U-statistic   p-value  \
29059 H vs L        1864.0  0.042352   
      H vs NL        469.0  0.021106   
5930  H vs L        2193.0  0.464216   
      H vs NL        501.0  0.050040   
      NL vs L       2769.0  0.007546   

                                         Shortened_Compound_Name  \
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...   
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV   
      H vs NL                             D-TRYPTOPHAN - 60.0 eV   
      NL vs L                             D-TRYPTOPHAN - 60.0 eV   

                                                   Compound_Name     comp1  
29059 H vs L   NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
      H vs NL  NCGC00385642-01!1,4-dihydroxyheptadec-16-en-2-...  2.594219  
5930  H vs L                              D-TRYPTOPHAN - 60.0 eV  2.271726  
      H vs NL                             D-TRYPTOPHAN - 60.0 eV  2.271726  
      NL vs L                             D-TRYPTOPHAN - 60.0 eV  2.271726

In [63]:
# Group by the primary index and collect the secondary index values
keep_dict = filtered_features.groupby(level=0).apply(lambda df: df.index.get_level_values(1).tolist()).to_dict()

In [64]:
# Function to check if the Feature and group combination is in keep_dict
def filter_rows(row):
    feature = row['ID']
    group = row['group']
    return feature in keep_dict and group in keep_dict[feature]

# Apply the filter function to VIP_filtered
VIP_filtered_subset = VIP_filtered[VIP_filtered.apply(filter_rows, axis=1)]

In [65]:
# Ensure Feature in VIP_filtered_subset and the primary index in filtered_features are the same type
VIP_filtered_subset['ID'] = VIP_filtered_subset['ID'].astype(int)
filtered_features.index = filtered_features.index.set_levels(
    filtered_features.index.levels[0].astype(int), level=0
)

# Create a MultiIndex mapping of (Feature, group) to p-value
p_value_mapping = filtered_features['p-value']

# Create a new column in VIP_filtered_subset for the p_value_bh
VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['ID', 'group']).index.map(p_value_mapping)

# Reset the index to return to the original structure if needed
VIP_filtered_subset.reset_index(drop=True, inplace=True)

# View the updated DataFrame
VIP_filtered_subset.head()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/947184108.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['ID'] = VIP_filtered_subset['ID'].astype(int)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/947184108.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  VIP_filtered_subset['p-value'] = VIP_filtered_subset.set_index(['ID', 'group']).index.map(p_value_mapping)


,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,...,group,VIP_direction,Feature_x,mz,Feature_y,RT,Feature,ClassyFire#most specific class,Shortened_Compound_Name,p-value
0,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,H vs NL,2.594219,29059,311.257397,29059,7.421430,29059,Fatty alcohols,C19H36O4 Fatty Alcohol,0.021106
1,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,H vs L,2.358565,29059,311.257397,29059,7.421430,29059,Fatty alcohols,C19H36O4 Fatty Alcohol,0.042352
2,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,NL vs L,-2.271726,5930,205.096918,5930,2.076001,5930,Alpha amino acids,Tryptophan,0.007546
3,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,...,NL vs L,1.947722,24392,428.372821,24392,6.156398,24392,Alpha amino acids and derivatives,Polysorbate derivative,0.080255
4,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,H vs NL,1.922795,5930,205.096918,5930,2.076001,5930,Alpha amino acids,Tryptophan,0.050040


In [66]:
# Map to categories from rclr boxplots
# category_map = {
#     # Fatty amides
#     "N-Oleoylethanolamine": "Fatty amides",

#     # Amino acids
#     "Phenylalanine": "Amino acids",
#     "Tyrosine": "Amino acids",
#     "Tryptophan": "Indole",
#     "(Iso)leucine": "Amino acids",
#     "Isoleucine": "Amino acids",
#     "Leucine": "Amino acids",

#     # Histidine derivatives
#     "Urocanic acid": "Histidine derivatives",

#     # Acyl carnitines
#     "Gln-C14:0": "Acyl carnitines",

#     # Dipeptides
#     "Glutamyltyrosine": "Dipeptides",
#     "Cyclo(his-pro)": "Dipeptides",
# }

# VIP_filtered_subset["Metabolite Class"] = (
#     VIP_filtered_subset["Shortened_Compound_Name"]
#     .map(category_map)
#)

# VIP_filtered_subset["Metabolite Class"] = (
#     VIP_filtered_subset["Metabolite Class"]
#     .fillna(VIP_filtered_subset["ClassyFire#most specific class"])
# )

VIP_filtered_subset = VIP_filtered_subset[
    VIP_filtered_subset["Shortened_Compound_Name"] != "Paracetamol"
]

VIP_filtered_subset

,ID,comp1,GroupContrib,importance,SpectrumID,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,...,group,VIP_direction,Feature_x,mz,Feature_y,RT,Feature,ClassyFire#most specific class,Shortened_Compound_Name,p-value
0,29059,2.594219,Acne_NL,-0.077379,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,H vs NL,2.594219,29059,311.257397,29059,7.421430,29059,Fatty alcohols,C19H36O4 Fatty Alcohol,0.021106
1,29059,2.358565,Acne_L,0.070163,CCMSLIB00005722082,spectra_filtered.mgf,GNPS-NIH-NATURALPRODUCTSLIBRARY_ROUND2_POSITIV...,0.612193,3807000,0,...,H vs L,2.358565,29059,311.257397,29059,7.421430,29059,Fatty alcohols,C19H36O4 Fatty Alcohol,0.042352
2,5930,2.271726,Acne_NL,-0.067610,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,NL vs L,-2.271726,5930,205.096918,5930,2.076001,5930,Alpha amino acids,Tryptophan,0.007546
3,24392,1.947722,Acne_L,0.057967,CCMSLIB00000577480,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.728038,591900,0,...,NL vs L,1.947722,24392,428.372821,24392,6.156398,24392,Alpha amino acids and derivatives,Polysorbate derivative,0.080255
4,5930,1.922795,Acne_NL,-0.057352,CCMSLIB00005883949,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.954860,4091900,0,...,H vs NL,1.922795,5930,205.096918,5930,2.076001,5930,Alpha amino acids,Tryptophan,0.050040
5,3794,1.824152,Acne_NL,-0.054289,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,306400,0,...,NL vs L,-1.824152,3794,311.123857,3794,1.203634,3794,Gamma-glutamyl amino acids,Glutamyltyrosine,0.109209
6,17958,1.823974,Acne_L,0.054284,CCMSLIB00012283450,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.776449,637600,0,...,NL vs L,1.823974,17958,373.127569,17958,5.024735,17958,5-O-methylated flavonoids,Sinensetin,0.881355
7,27758,1.814812,Acne_NL,-0.054131,CCMSLIB00003137348,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.814365,282600,0,...,H vs NL,1.814812,27758,326.305056,27758,7.132356,27758,N-acyl amines,N-Oleoylethanolamine,0.179420
8,7778,1.729160,Acne_NL,-0.051577,CCMSLIB00000579531,spectra_filtered.mgf,CASMI.mgf,0.968451,17502000,0,...,H vs NL,1.729160,7778,275.048184,7778,2.761023,7778,Phenylbenzimidazoles,Phenylbenzimidazole sulfonic acid,0.584706
9,3794,1.716047,Healthy,-0.051049,CCMSLIB00005747606,spectra_filtered.mgf,MASSBANK.mgf,0.844589,306400,0,...,H vs L,-1.716047,3794,311.123857,3794,1.203634,3794,Gamma-glutamyl amino acids,Glutamyltyrosine,0.680104


### Plot VIP figure

In [67]:
# Set up the figure
fig, axes = plt.subplots(
    1, 3, figsize=(20, 6),
    gridspec_kw={"wspace": 0.75}
)

unique_groups = ["H vs L", "H vs NL", "NL vs L"]

# Get all unique class categories
unique_classes = VIP_filtered_subset['ClassyFire#most specific class'].dropna().unique()
# unique_classes = VIP_filtered_subset['Metabolite Class'].dropna().unique()

palette = sns.color_palette("hls", len(unique_classes))
color_map = dict(zip(unique_classes, palette))

fig.suptitle('Top VIP Features Across Groups (VIP > 1)', fontsize=20, y=1.025)

for i, group in enumerate(unique_groups):
    group_data = VIP_filtered_subset[VIP_filtered_subset['group'] == group].copy()

    # Sort by abs(VIP score) descending
    group_data = group_data.sort_values('VIP_direction', key=lambda x: x.abs(), ascending=False).reset_index(drop=True)

    vip_scores = group_data['VIP_direction']
    
    # Format m/z_RT for y-axis labels
    # ids = group_data.apply(lambda row: f"{row['mz']:.4f}_{row['RT']:.2f}", axis=1)
    ids = group_data['Shortened_Compound_Name']

    classes = group_data['ClassyFire#most specific class']
    dot_colors = classes.map(color_map).fillna('gray')

    # Plot
    axes[i].scatter(
        vip_scores,
        range(len(group_data)),
        color=dot_colors,
        s=200
    )

    # VIP cutoff lines
    axes[i].axvline(x=1, color='gray', linestyle='--', linewidth=1.5)
    axes[i].axvline(x=-1, color='gray', linestyle='--', linewidth=1.5)

    # Y-axis = m/z_RT labels
    axes[i].set_yticks(range(len(ids)))
    axes[i].set_yticklabels(ids, fontsize=10)
    axes[i].invert_yaxis()
    axes[i].set_xlabel('VIP Scores', fontsize=12)
    # axes[i].set_ylabel('m/z_RT', fontsize=12, labelpad=10)

    custom_titles = {
        "H vs L": "Healthy vs Lesional",
        "H vs NL": "Healthy vs Non-lesional",
        "NL vs L": "Non-lesional vs Lesional"
    }
    axes[i].set_title(custom_titles.get(group, group), fontsize=14, pad=10)

# Legend for ClassyFire classes
legend_handles = [plt.Line2D([0], [0], marker='o', color=color, linestyle='', markersize=10, label=cls)
                  for cls, color in color_map.items()]



fig.legend(handles=legend_handles, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.24),
           fontsize=12, title='Metabolite Class', title_fontsize=14, frameon=False)

# Final layout and save
plt.tight_layout(rect=[0, 0.15, 0.9, 1])
plt.savefig("../Figures/Main/Figure_5/VIP_combined_plot_by_mzRT_sorted_colored_most-specific-class.svg", dpi=600, bbox_inches='tight')
# plt.savefig("../Figures/Main/Figure_5/VIP_combined_plot_by_mzRT_sorted_colored_metabolite-class.svg", dpi=600, bbox_inches='tight')

plt.savefig("../Figures/Main/Figure_5/VIP_combined_plot_by_mzRT_sorted_colored_most-specific-class.png", dpi=600, bbox_inches='tight')
# plt.savefig("../Figures/Main/Figure_5/VIP_combined_plot_by_mzRT_sorted_colored_metabolite-class.png", dpi=600, bbox_inches='tight')

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11648/1936888437.py:69: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.15, 0.9, 1])


## Subset metabolomics top VIP table for MMVEC analysis

In [68]:
metaB_feats_to_keep = [655, 1368, 2969, 5930, 23297, 24392, 29059]
metaB_table_subset = metaB_table_subset[metaB_feats_to_keep]

metaB_table_subset


,655,1368,2969,5930,23297,24392,29059
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.00,611709.700
LAMI.RD308.D2.C2,3065.1714,1669351.50,750870.100,442653.660,0.0000,0.00,231525.840
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.69,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.92,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.00,427202.060
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.00,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.00,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.00,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.00,55986.290


In [69]:
# Save as csv
metaB_table_subset.to_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs_final.csv')